In [11]:
!pip install detoxify
!pip install better_profanity
!pip install clean-text

In [12]:
from detoxify import Detoxify

# initialise toxicity detection pipeline
toxicity_detector = Detoxify('original')

def detect_toxicity(text):
    results = toxicity_detector.predict(text)
    return results

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [13]:
# Test detect_toxicity function

text1 = 'You are such a damn fool!'

toxicity_results = detect_toxicity(text1)

print('Toxicity :',toxicity_results['toxicity'])
print('Toxicity Results :\n',toxicity_results)

Toxicity : 0.9917719
Toxicity Results :
 {'toxicity': np.float32(0.9917719), 'severe_toxicity': np.float32(0.06865552), 'obscene': np.float32(0.9102149), 'threat': np.float32(0.0012524179), 'insult': np.float32(0.9419541), 'identity_attack': np.float32(0.009397822)}


In [14]:
from better_profanity import profanity
from detoxify import Detoxify

# Initialize profanity
profanity.load_censor_words()

def filter_toxic_content(text):
  # Check if text is toxic

  toxic_score = 0.9
  is_toxic = toxicity_detector.predict(text)['toxicity'] >= toxic_score

  if is_toxic:
    print(f'Toxic content detected with score :{toxic_score}')
    # Use profanity filter censor toxic words
    filtered_text = profanity.censor(text)
    return filtered_text
  else:
    return text

In [15]:
text2 = 'You are such a moran you stupid!'

filtered_output1 = filter_toxic_content(text2)

print('Filtered Text:',filtered_output1)

Toxic content detected with score :0.9
Filtered Text: You are such a moran you ****!


In [16]:
import re
from cleantext import clean

# Detect PII data
def detect_pii(text):
  # Remove sensitive info: emails, phone numbers
  cleaned_text = clean(
      text,
      no_emails = True,
      no_urls = True,
      no_phone_numbers = True,
      no_numbers = True
  )
  return cleaned_text

In [17]:
text3 = """
Hello, please reach out to me at john.doe@example.com or call me at 555-123-4567.
My credit card number is 4111 1111 1111 1111.
"""

print('Original Text:\n', text3)
print("\n Filtered Text (PII removed):\n", detect_pii(text3))

Original Text:
 
Hello, please reach out to me at john.doe@example.com or call me at 555-123-4567.
My credit card number is 4111 1111 1111 1111.


 Filtered Text (PII removed):
 hello, please reach out to me at <email> or call me at <phone>.
my credit card number is <number> <phone> <number>.


In [18]:
from better_profanity import profanity

# Sample text to filter
text4 = """
Hey, you are a really bad person! stop being so rude.
"""

# Define a custom list of words you want to censor
custom_bad_words = ['bad','rude','person']

In [19]:
# Load the custom words
profanity.load_censor_words(custom_bad_words)

# Filter text text
filtered_outpu2 = profanity.censor(text4)

print('Original Text:\n', text4)
print('\nFiltered Text:\n', filtered_outpu2)

Original Text:
 
Hey, you are a really bad person! stop being so rude.


Filtered Text:
 
Hey, you are a really **** ****! stop being so ****.



In [20]:
from cleantext import clean
from better_profanity import profanity

def mask_match(match):
  word = match.group(0)
  if len(word) < 2:
    return word
  return word[0] + ('*' * (len(word) - 2)) + word[-1]

def censor_severity(text):
  # List of milder and profanitites
  mild_profanities = ['damn','crap']
  severe_profanities = ['hell','stupid']

  # Replace emails and URLs
  text = clean(
      text,
      no_emails=True,
      no_urls=True,
      no_phone_numbers=True,
      no_numbers=True
  )

  pattern = r'\b('+'|'.join(map(re.escape,mild_profanities))+r')\b'

  # Handle severe profanity first (complete masking)
  profanity.load_censor_words(severe_profanities)
  text = profanity.censor(text)

  # Handle mild profanity (partial masking)
  profanity.load_censor_words(mild_profanities)

  for word in mild_profanities:
    text = re.sub(pattern, mask_match, text)
  return text

text5 = 'This is a crap and a damn hell of a situation! You stupid email me at john.doe@example.com'
print(censor_severity(text5))

this is a c**p and a d**n **** of a situation! you **** email me at <email>
